# Notebook 1: DepMap Data Loading and Preprocessing

This notebook downloads and preprocesses the Cancer Dependency Map (DepMap) data
for context-specific fitness gene analysis.

**Data sources:**
- **DepMap Public**: CRISPR gene-effect scores (Chronos algorithm)
- **Cell Model Passport / CCLE**: Cell line metadata and cancer type annotations

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../../data'
os.makedirs(DATA_DIR, exist_ok=True)

# DepMap release version
DEPMAP_RELEASE = '26Q1'

## 1.1 Download DepMap Data

Download the following files from [DepMap Portal](https://depmap.org/portal/download/):

1. **CRISPRGeneEffect.csv** — Gene-level dependency scores (Chronos-corrected).
   Each value represents the fitness effect of knocking out a gene in a cell line; negative values indicate reduced fitness (essentiality).

2. **Model.csv** — Cell line metadata including lineage, cancer type annotations, and subtype information.

All downloaded files should be in the `../../data/` directory.

In [ ]:
# ── Verify required files exist ─────────────────────────────
REQUIRED_FILES = {
    'gene_effect': 'CRISPRGeneEffect.csv',
    'model_info':  'Model.csv',
}

for label, fname in REQUIRED_FILES.items():
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  [OK]  {fname} ({size_mb:.1f} MB)')
    else:
        print(f'  [!!]  {fname} — NOT FOUND.')

  [OK]  CRISPRGeneEffect.csv (440.6 MB)
  [OK]  Model.csv (0.7 MB)


## 1.2 Load Gene-Effect Matrix

The CRISPR gene-effect matrix contains Chronos-corrected dependency scores.
Rows are cell lines (identified by DepMap IDs) and columns are genes
(formatted as `GENENAME (ENTREZ_ID)`).

- Score ≈ 0: no fitness effect (gene is non-essential)
- Score << 0: strong negative fitness effect (gene is essential)
- Conventional threshold: score ≤ −0.5 to −0.6 indicates dependency

In [ ]:
# ── Load gene-effect matrix ─────────────────────────────────
gene_effect = pd.read_csv(
    os.path.join(DATA_DIR, REQUIRED_FILES['gene_effect']),
    index_col=0
)

print(f'Gene-effect matrix shape: {gene_effect.shape}')
print(f'  Cell lines: {gene_effect.shape[0]}')
print(f'  Genes:      {gene_effect.shape[1]}')
print(f'  Missing values: {gene_effect.isna().sum().sum():,} '
      f'({100*gene_effect.isna().sum().sum()/(gene_effect.shape[0]*gene_effect.shape[1]):.1f}%)')

# Preview
gene_effect.iloc[:5, :5]

Gene-effect matrix shape: (1208, 18531)
  Cell lines: 1208
  Genes:      18531
  Missing values: 887,151 (4.0%)


,A1BG (1),A1CF (29974),A2M (2),A2ML1 (144568),A3GALT2 (127550)
ACH-000029,0.000151,0.000541,-0.046740,0.040447,-0.188685
ACH-000030,0.169029,-0.182773,-0.023095,0.068845,-0.037532
ACH-000074,-0.087069,-0.054984,0.211763,0.421891,0.013034
ACH-000090,0.205441,-0.138211,-0.140932,-0.139838,-0.036163
ACH-000093,-0.133302,0.086477,-0.224933,0.024604,0.267048


In [ ]:
# ── Clean gene column names ───────────────────────────────
# Columns are formatted as 'GENE (ENTREZ_ID)'; extract just the gene symbol.

def parse_gene_name(col):
    """Extract gene symbol from DepMap column format 'GENE (12345)'."""
    if ' (' in col:
        return col.split(' (')[0]
    return col

gene_names = [parse_gene_name(c) for c in gene_effect.columns]

# Check for duplicate gene symbols (rare but possible)
from collections import Counter
dupes = [g for g, count in Counter(gene_names).items() if count > 1]
if dupes:
    print(f'Warning: {len(dupes)} duplicate gene symbols found: {dupes[:10]}')
    print('Keeping original column names to avoid ambiguity.')
else:
    gene_effect.columns = gene_names
    print(f'Renamed {len(gene_names)} columns to gene symbols (no duplicates).')

# Verify key genes are present
key_genes = ['ZMYM3', 'SMARCB1', 'SMARCA4', 'EZH2', 'BRAF', 'WRN']
for g in key_genes:
    present = g in gene_effect.columns
    print(f'  {g}: {"found" if present else "NOT FOUND"}')

Renamed 18531 columns to gene symbols (no duplicates).
  ZMYM3: found
  SMARCB1: found
  SMARCA4: found
  EZH2: found
  BRAF: found
  WRN: found


## 1.3 Load Cell Line Metadata and Annotate Cancer Types

In [ ]:
# ── Load cell line metadata ─────────────────────────────────
model_info = pd.read_csv(
    os.path.join(DATA_DIR, REQUIRED_FILES['model_info'])
)

print(f'Model metadata: {model_info.shape[0]} cell lines, {model_info.shape[1]} fields')
print(f'\nAvailable columns:')
for c in model_info.columns:
    print(f'  - {c}')

Model metadata: 2154 cell lines, 49 fields

Available columns:
  - ModelID
  - PatientID
  - CellLineName
  - StrippedCellLineName
  - DepmapModelType
  - OncotreeLineage
  - OncotreePrimaryDisease
  - OncotreeSubtype
  - OncotreeCode
  - PatientSubtypeFeatures
  - RRID
  - Age
  - AgeCategory
  - Sex
  - PatientRace
  - PrimaryOrMetastasis
  - SampleCollectionSite
  - SourceType
  - SourceDetail
  - CatalogNumber
  - ModelType
  - TissueOrigin
  - ModelDerivationMaterial
  - ModelTreatment
  - PatientTreatmentStatus
  - PatientTreatmentType
  - PatientTreatmentDetails
  - Stage
  - StagingSystem
  - PatientTumorGrade
  - PatientTreatmentResponse
  - GrowthPattern
  - OnboardedMedia
  - FormulationID
  - SerumFreeMedia
  - PlateCoating
  - EngineeredModel
  - EngineeredModelDetails
  - CulturedResistanceDrug
  - PublicComments
  - CCLEName
  - HCMIID
  - PediatricModelType
  - ModelAvailableInDbgap
  - ModelSubtypeFeatures
  - WTSIMasterCellID
  - SangerModelID
  - COSMICID
  - ModelID

In [ ]:
# ── Build cancer indication mapping ───────────────────────
# The 'OncotreeLineage' or 'OncotreeSubtype' columns provide cancer type
# annotations. We use OncotreeSubtype to better stratify specific cancer
# indications, including AT/RT.

ANNOT_COL = 'OncotreeSubtype'
print(f'\nUsing annotation column: {ANNOT_COL}')

# Show all unique cancer types
indication_counts = model_info[ANNOT_COL].value_counts()
print(f'\nTotal unique indications: {len(indication_counts)}')
print(f'\nTop 20 indications by cell line count:')
print(indication_counts.head(20).to_string())


Using annotation column: OncotreeSubtype

Total unique indications: 254

Top 20 indications by cell line count:
OncotreeSubtype
Melanoma                               109
Lung Adenocarcinoma                     91
Small Cell Lung Cancer                  82
Colon Adenocarcinoma                    74
Glioblastoma, IDH-Wildtype              67
Pancreatic Adenocarcinoma               64
Oral Cavity Squamous Cell Carcinoma     62
Neuroblastoma                           53
Ewing Sarcoma                           51
Acute Myeloid Leukemia                  42
Breast Invasive Ductal Carcinoma        40
Matched Normal                          40
Bladder Urothelial Carcinoma            35
Plasma Cell Myeloma                     34
Stomach Adenocarcinoma                  34
Osteosarcoma                            34
B-Cell Acute Lymphoblastic Leukemia     33
Diffuse Large B-Cell Lymphoma, NOS      32
Lung Squamous Cell Carcinoma            32
Esophageal Squamous Cell Carcinoma      30


In [ ]:
# ── Find AT/RT cell lines ─────────────────────────────────
# Search for AT/RT-related annotations across multiple columns

atrt_keywords = ['rhabdoid', 'atrt', 'atypical teratoid', 'AT/RT',
                 'Atypical Teratoid']

def find_atrt_lines(df, search_cols):
    """Find cell lines matching AT/RT-related keywords."""
    mask = pd.Series(False, index=df.index)
    for col in search_cols:
        if col in df.columns:
            col_str = df[col].fillna('').str.lower()
            for kw in atrt_keywords:
                mask |= col_str.str.contains(kw.lower())
    return mask

search_cols = ['OncotreeSubtype', 'OncotreePrimaryDisease', 'OncotreeLineage',
               'OncotreeCode', 'CellLineName']
atrt_mask = find_atrt_lines(model_info, search_cols)

atrt_lines = model_info[atrt_mask]
print(f'Found {len(atrt_lines)} AT/RT cell lines:')
display_cols = ['ModelID', 'CellLineName', 'OncotreeLineage',
                'OncotreePrimaryDisease', 'OncotreeSubtype']
display_cols = [c for c in display_cols if c in model_info.columns]
if len(atrt_lines) > 0:
    print(atrt_lines[display_cols].to_string())

Found 22 AT/RT cell lines:
         ModelID  CellLineName OncotreeLineage OncotreePrimaryDisease                   OncotreeSubtype
94    ACH-000096         G-401          Kidney        Rhabdoid Cancer                   Rhabdoid Cancer
158   ACH-000160         BT-12       CNS/Brain        Embryonal Tumor  Atypical Teratoid/Rhabdoid Tumor
170   ACH-000172         TM-87     Soft Tissue        Rhabdoid Cancer        Extrarenal Rhabdoid Cancer
199   ACH-000201         A-204          Kidney        Rhabdoid Cancer                   Rhabdoid Cancer
373   ACH-000375         G-402          Kidney        Rhabdoid Cancer                   Rhabdoid Cancer
529   ACH-000533  NCI-H2004 RT          Kidney        Rhabdoid Cancer                   Rhabdoid Cancer
593   ACH-000597       TTC-709     Soft Tissue        Rhabdoid Cancer        Extrarenal Rhabdoid Cancer
603   ACH-000607         KYM-1          Kidney        Rhabdoid Cancer                   Rhabdoid Cancer
1008  ACH-001020         BT-16       

In [ ]:
# ── Build the analysis-ready dataset ──────────────────────
# Merge gene-effect scores with cancer type annotations.

# Identify the ID column in model_info that matches gene_effect index
id_col = 'ModelID'

# Filter to cell lines present in both datasets
common_ids = gene_effect.index.intersection(model_info[id_col])
print(f'Cell lines in gene-effect matrix: {gene_effect.shape[0]}')
print(f'Cell lines in metadata:           {model_info.shape[0]}')
print(f'Cell lines in both (intersection): {len(common_ids)}')

# Create annotation lookup
id_to_indication = model_info.set_index(id_col)[ANNOT_COL].to_dict()

# Add indication labels to gene-effect matrix
gene_effect_annotated = gene_effect.loc[common_ids].copy()
gene_effect_annotated.insert(0, 'indication',
                              [id_to_indication.get(idx, 'Unknown') for idx in common_ids])

# Remove cell lines with missing indication
gene_effect_annotated = gene_effect_annotated[
    gene_effect_annotated['indication'].notna() &
    (gene_effect_annotated['indication'] != 'Unknown')
]

print(f'\nFinal analysis dataset: {gene_effect_annotated.shape[0]} cell lines, '
      f'{gene_effect_annotated.shape[1]-1} genes')
print(f'Unique indications: {gene_effect_annotated["indication"].nunique()}')

Cell lines in gene-effect matrix: 1208
Cell lines in metadata:           2154
Cell lines in both (intersection): 1208

Final analysis dataset: 1208 cell lines, 18531 genes
Unique indications: 169


In [ ]:
# ── Merge all rhabdoid-related indications ────────────────
RHABDOID_LABEL = 'Atypical Teratoid/Rhabdoid Tumor'

# Find all indication values containing "rhabdoid" (case-insensitive)
rhabdoid_mask = gene_effect_annotated['indication'].str.contains(
    'rhabdoid', case=False, na=False
)

original_labels = gene_effect_annotated.loc[rhabdoid_mask, 'indication'].unique()
print(f'Found {rhabdoid_mask.sum()} cell lines across {len(original_labels)} rhabdoid-related indications:')
for label in original_labels:
    n = (gene_effect_annotated['indication'] == label).sum()
    print(f'  - "{label}" ({n} cell lines)')

# Relabel them all
gene_effect_annotated.loc[rhabdoid_mask, 'indication'] = RHABDOID_LABEL
print(f'\nMerged into: "{RHABDOID_LABEL}" ({rhabdoid_mask.sum()} cell lines total)')

Found 16 cell lines across 3 rhabdoid-related indications:
  - "Extrarenal Rhabdoid Cancer" (5 cell lines)
  - "Atypical Teratoid/Rhabdoid Tumor" (6 cell lines)
  - "Rhabdoid Cancer" (5 cell lines)

Merged into: "Atypical Teratoid/Rhabdoid Tumor" (16 cell lines total)


In [ ]:
# ── Summary statistics ────────────────────────────────────
indication_summary = (
    gene_effect_annotated
    .groupby('indication')
    .size()
    .sort_values(ascending=False)
    .reset_index(name='n_cell_lines')
)

print(f'Indication summary ({len(indication_summary)} indications):\n')
print(indication_summary.to_string())

Indication summary (167 indications):

                                                                                              indication  n_cell_lines
0                                                                                               Melanoma            55
1                                                                                    Lung Adenocarcinoma            53
2                                                                    Oral Cavity Squamous Cell Carcinoma            53
3                                                                             Glioblastoma, IDH-Wildtype            50
4                                                                                   Colon Adenocarcinoma            48
5                                                                              Pancreatic Adenocarcinoma            46
6                                                                                          Neuroblastoma            41
7        

In [ ]:
# ── Save preprocessed data ────────────────────────────────
gene_effect_annotated.to_csv(os.path.join(DATA_DIR, 'gene_effect_annotated.csv'))
indication_summary.to_csv(os.path.join(DATA_DIR, '01_indication_summary.csv'), index=False)

print('Saved:')
print(f'  gene_effect_annotated.csv ({gene_effect_annotated.shape})')
print(f'  indication_summary.csv ({indication_summary.shape})')
print('\nPreprocessing complete. Proceed to Notebook 2.')

Saved:
  gene_effect_annotated.csv ((1208, 18532))
  indication_summary.csv ((167, 2))

Preprocessing complete. Proceed to Notebook 2.
